# Mental Health Score Prediction

## Professional, Leakage-Free ML Workflow

Yeh notebook modern scikit-learn tools use karta hai:

- `ColumnTransformer` — har column type par sahi preprocessing apply karta hai
- `Pipeline` — preprocessing + model ko ek object mein combine karta hai
- `OneHotEncoder(min_frequency=...)` — rare categories ko automatically group karta hai
- `cross_validate` — model comparison ko reliable banata hai
- `joblib` — trained pipeline ko deployment ke liye save karta hai

### Workflow

1. Original dataset load
2. Basic checks + EDA
3. Rule-based cleaning (duplicates, negative values)
4. Train/Test split — **preprocessing se pehle**
5. `ColumnTransformer` sirf training data par fit
6. Model pipelines train + compare
7. Best pipeline save

> **Golden Rule:** Test data ko kabhi `fit` nahi karna. Sirf `transform` / `predict` karna hai.

## 1. Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gzip
import pickle
import warnings

warnings.filterwarnings('ignore', message='Found unknown categories')

from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Original Dataset Load

Yahan **original CSV** load ho rahi hai, koi encoded file nahi.

In [ ]:
df = pd.read_csv('Student_Social_Media_And_Mental_Health_Impact.csv')

print("Original Shape:", df.shape)

### 2.1 Columns aur Data Types

In [ ]:
print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

### 2.2 Missing Values

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("\nTotal Missing Values:", df.isnull().sum().sum())

### 2.3 Duplicates aur Summary Statistics

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

print("\nSummary Statistics:")
print(df.describe())

In [ ]:
print("Complete Summary (including categorical columns):")
print(df.describe(include='all'))

## 3. Exploratory Data Analysis (EDA)

Yeh plots data ko samajhne ke liye hain — preprocessing decisions sirf training data se lenge.

In [ ]:
# Target Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Mental_Health_Score'], bins=20, kde=True)
plt.title('Distribution of Mental Health Score')
plt.xlabel('Mental Health Score')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
correlation = df.select_dtypes(include='number').corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Stress Level vs Mental Health Score
plt.figure(figsize=(8, 5))
sns.boxplot(x='Stress_Level', y='Mental_Health_Score', data=df)
plt.title('Stress Level vs Mental Health Score')
plt.xlabel('Stress Level')
plt.ylabel('Mental Health Score')
plt.show()

In [ ]:
# Social Media Usage vs Mental Health Score
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='Avg_Daily_Usage_Hours',
    y='Mental_Health_Score',
    data=df
)
plt.title('Social Media Usage vs Mental Health Score')
plt.xlabel('Average Daily Usage Hours')
plt.ylabel('Mental Health Score')
plt.show()

In [ ]:
# Sleep Hours vs Mental Health Score
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='Sleep_Hours_Per_Night',
    y='Mental_Health_Score',
    data=df
)
plt.title('Sleep Hours vs Mental Health Score')
plt.xlabel('Sleep Hours Per Night')
plt.ylabel('Mental Health Score')
plt.show()

In [ ]:
# Most Used Social Media Platforms
plt.figure(figsize=(12, 6))
sns.countplot(
    data=df,
    x='Most_Used_Platform',
    order=df['Most_Used_Platform'].value_counts().index
)
plt.title('Most Used Social Media Platforms')
plt.xlabel('Social Media Platform')
plt.ylabel('Number of Students')
plt.xticks(rotation=45)
plt.show()

## 4. Data Cleaning

Split se pehle sirf **rule-based** cleaning kar rahe hain:

1. Exact duplicate rows remove
2. Negative `Physical_Activity_Hours` ko `0` se replace

Yeh steps kisi statistical learning par depend nahi karte, isliye leakage create nahi karte.

### 4.1 Duplicate Rows Remove

In [ ]:
print("Duplicates Before Cleaning:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicates After Cleaning:", df.duplicated().sum())
print("Shape After Duplicate Removal:", df.shape)

### 4.2 Negative Values Fix

In [ ]:
print("Negative Physical Activity Values Before:",
      (df['Physical_Activity_Hours'] < 0).sum())

df.loc[df['Physical_Activity_Hours'] < 0, 'Physical_Activity_Hours'] = 0

print("Negative Physical Activity Values After:",
      (df['Physical_Activity_Hours'] < 0).sum())

print("Minimum Physical Activity Hours:", df['Physical_Activity_Hours'].min())

### 4.3 Cleaning Verification

In [ ]:
print("Remaining Duplicates:", df.duplicated().sum())
print("Negative Physical Activity Values:",
      (df['Physical_Activity_Hours'] < 0).sum())
print("Dataset Shape After Cleaning:", df.shape)

## 5. Train/Test Split (Preprocessing se PEHLE)

**Kyun split pehle?**

- Imputer, encoder aur scaler ke parameters sirf training data se seekhne chahiye.
- Agar poore dataset par fit karenge to test data ki information train mein chali jayegi — isay **data leakage** kehte hain.
- Leakage se test performance artificially achi lagti hai, real world mein model fail karta hai.

### 5.1 Features aur Target Separate

In [ ]:
X = df.drop('Mental_Health_Score', axis=1)
y = df['Mental_Health_Score']

print("X Shape:", X.shape)
print("y Shape:", y.shape)

### 5.2 80/20 Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### 5.3 Split Shapes Check

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

### 5.4 Skewness Check (Sirf Training Data)

Skewness ka faisla training data se lena chahiye. Agar koi column strongly skewed hota (`> 0.5`) to log transform add karte. Yahan sab values `0.5` se kam hain, isliye `log1p` ki zaroorat nahi.

> Note: Tutorial mein `Study_Hours` par `log1p` suggest kiya gaya tha, lekin actual skewness `0.436` hai. Best practice hai data-driven decision lena, blindly transform nahi lagana.

In [ ]:
numeric_cols = X_train.select_dtypes(include='number').columns
train_skew = X_train[numeric_cols].skew().sort_values(ascending=False)

print("Skewness (Training Data Only):")
print(train_skew)

print("\nDecision: max skew < 0.5, so no log transform needed.")

## 6. Modern Preprocessing with `ColumnTransformer`

Ab manual column-by-column preprocessing nahi karenge. Ek `ColumnTransformer` banayenge jo har column type par sahi treatment apply karega.

| Branch | Columns | Steps |
|---|---|---|
| Numeric | Age, usage, unlocks, study, activity, sleep | `SimpleImputer(median)` → `StandardScaler` |
| Ordinal | Academic_Level, Stress_Level | `SimpleImputer(most_frequent)` → `OrdinalEncoder` with explicit order |
| Nominal | Gender, Country, Platform, Purpose | `SimpleImputer(most_frequent)` → `OneHotEncoder(min_frequency=0.01)` |

`min_frequency=0.01` rare categories (jaise rare countries) ko automatically ek infrequent category mein group kar deta hai — manual top-10 country grouping ki zaroorat nahi.

`SimpleImputer` har branch mein hai as a **safety net**, chahe abhi missing values `0` hain. Production data aur API requests hamesha clean nahi hoti.

### 6.1 Column Groups Define

In [ ]:
numeric_features = [
    'Age',
    'Avg_Daily_Usage_Hours',
    'Daily_Unlocks',
    'Study_Hours',
    'Physical_Activity_Hours',
    'Sleep_Hours_Per_Night'
]

ordinal_features = ['Academic_Level', 'Stress_Level']

nominal_features = [
    'Gender',
    'Country',
    'Most_Used_Platform',
    'Purpose_Of_Use'
]

academic_order = ['High School', 'Undergraduate', 'Graduate']
stress_order = ['Low', 'Medium', 'High', 'Very High']

print("Numeric :", numeric_features)
print("Ordinal :", ordinal_features)
print("Nominal :", nominal_features)

### 6.2 Numeric Pipeline

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

### 6.3 Ordinal Pipeline

`OrdinalEncoder` ko explicit order diya gaya hai, taake numbers ka meaning sahi rahe.

In [ ]:
ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[academic_order, stress_order],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

### 6.4 Nominal Pipeline

`handle_unknown='infrequent_if_exist'` production ke liye best hai — API se koi naya category aaye to error nahi aayega, woh infrequent category mein chala jayega.

Test set mein kuch categories train mein nahi thin, isliye sklearn expected warning deta hai. Humne unhe infrequent category mein map kiya hai aur warning suppress kar di hai.

In [ ]:
nominal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=0.01,
        drop='first',
        sparse_output=False
    ))
])

### 6.5 Combine with `ColumnTransformer`

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('ord', ordinal_pipeline, ordinal_features),
        ('nom', nominal_pipeline, nominal_features)
    ],
    remainder='drop'
)

print(preprocessor)

### 6.6 Fit on Training Data Only

`fit_transform` sirf `X_train` par, aur `transform` sirf `X_test` par. Yeh cell inspection ke liye hai — model training ke waqt poora pipeline use karenge.

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_prepared = pd.DataFrame(
    X_train_prepared,
    columns=feature_names,
    index=X_train.index
)

X_test_prepared = pd.DataFrame(
    X_test_prepared,
    columns=feature_names,
    index=X_test.index
)

print("Prepared train shape:", X_train_prepared.shape)
print("Prepared test shape :", X_test_prepared.shape)
print("\nFeature names:")
print(feature_names)

In [ ]:
print("First 5 rows of prepared training data:")
print(X_train_prepared.head())

## 7. Build Model Pipelines

Ab preprocessing aur model ko ek `Pipeline` mein combine kar rahe hain.

**Why Pipeline?**

- `.fit()` ek hi call mein preprocessing + model training karta hai
- `.predict()` raw data par automatically wohi preprocessing apply karta hai
- Cross-validation mein preprocessing sirf training folds par fit hota hai — no leakage
- Deployment mein ek hi object load karna hota hai

### 7.1 Define Models

Do models compare kar rahe hain:

- **Linear Regression** — simple, interpretable baseline
- **Random Forest Regressor** — non-linear patterns ke liye strong model

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

pipelines = {
    name: Pipeline(steps=[
        ('preprocessor', clone(preprocessor)),
        ('regressor', model)
    ])
    for name, model in models.items()
}

print(pipelines['Linear Regression'])

### 7.2 Train aur Test Performance

In [ ]:
results = []

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values('RMSE')
print(results_df.to_string(index=False))

### 7.3 Cross-Validation (Fair Comparison)

Test set par ek hi split se comparison thoda luck-dependent hota hai. 5-fold cross-validation zyada reliable estimate deta hai.

In [ ]:
cv_results = []

for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=[
            'r2',
            'neg_mean_absolute_error',
            'neg_root_mean_squared_error'
        ]
    )

    cv_results.append({
        'Model': name,
        'CV R2': scores['test_r2'].mean(),
        'CV MAE': -scores['test_neg_mean_absolute_error'].mean(),
        'CV RMSE': -scores['test_neg_root_mean_squared_error'].mean()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values('CV RMSE')
print(cv_results_df.to_string(index=False))

### 7.4 Best Baseline Model — Actual vs Predicted

Model selection **CV RMSE** par kar rahe hain, test set par nahi. Test set sirf final performance report karne ke liye hai.

In [ ]:
best_model_name = cv_results_df.iloc[0]['Model']
best_pipeline = pipelines[best_model_name]
y_pred_best = best_pipeline.predict(X_test)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=y_pred_best, alpha=0.5)

line_min = min(y_test.min(), y_pred_best.min())
line_max = max(y_test.max(), y_pred_best.max())
plt.plot([line_min, line_max], [line_min, line_max], 'r--')

plt.title(f'Actual vs Predicted — {best_model_name}')
plt.xlabel('Actual Mental Health Score')
plt.ylabel('Predicted Mental Health Score')
plt.show()

## 8. Hyperparameter Tuning — Random Forest

Ab Random Forest ke hyperparameters tune karenge. Hum `RandomizedSearchCV` use kar rahe hain, `GridSearchCV` nahi — kyunki full grid search har combination test karta hai jo bahut slow hota hai. `RandomizedSearchCV` fixed number of combinations sample karta hai aur strong parameters find karta hai.

**Pipeline prefix rule:** Pipeline ke andar model step ka naam `regressor` hai, isliye har parameter ka prefix `regressor__` hoga. Example: `regressor__n_estimators`. Prefix ke bina `RandomizedSearchCV` ko pata nahi chalega ke parameter kis step ka hai.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'regressor__n_estimators': [100, 200, 300, 500],
    'regressor__max_depth': [None, 10, 20, 30, 40],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2', 1.0],
    'regressor__bootstrap': [True, False]
}

rf_pipeline = Pipeline(steps=[
    ('preprocessor', clone(preprocessor)),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=1))
])

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train, y_train)

print("Best CV RMSE:", round(-random_search.best_score_, 4))

print("\nBest parameters:")
for key, value in random_search.best_params_.items():
    print(f"  {key}: {value}")

### 8.1 Tuned Model Performance

Ab tuned model ko test set par evaluate kar rahe hain aur baseline Random Forest se compare kar rahe hain.

In [ ]:
tuned_pipeline = random_search.best_estimator_
y_pred_tuned = tuned_pipeline.predict(X_test)

tuned_metrics = {
    'Model': 'Random Forest (Tuned)',
    'MAE': mean_absolute_error(y_test, y_pred_tuned),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_tuned)),
    'R2': r2_score(y_test, y_pred_tuned)
}

baseline_rf = results_df[results_df['Model'] == 'Random Forest'].iloc[0]

comparison_df = pd.DataFrame([
    {
        'Model': 'Random Forest (Baseline)',
        'MAE': baseline_rf['MAE'],
        'RMSE': baseline_rf['RMSE'],
        'R2': baseline_rf['R2']
    },
    tuned_metrics
])

print(comparison_df.to_string(index=False))

### 8.2 Final Model Selection

Model selection **CV RMSE** par kar rahe hain, test set par nahi — test set sirf final report ke liye hai.

In [ ]:
baseline_cv_rmse = cv_results_df.loc[
    cv_results_df['Model'] == 'Random Forest', 'CV RMSE'
].iloc[0]

tuned_cv_rmse = -random_search.best_score_

print(f"Baseline RF CV RMSE: {baseline_cv_rmse:.4f}")
print(f"Tuned RF CV RMSE   : {tuned_cv_rmse:.4f}")

if tuned_cv_rmse < baseline_cv_rmse:
    final_pipeline = tuned_pipeline
    final_model_name = 'Random Forest (Tuned)'
else:
    final_pipeline = pipelines['Random Forest']
    final_model_name = 'Random Forest (Baseline)'

print(f"\nSelected final model: {final_model_name}")

## 9. Model Evaluation

Ab final tuned model ko **untouched test set** par evaluate karenge. Yeh woh data hai jo preprocessing fit, model training, ya hyperparameter tuning mein kabhi use nahi hua.

### 9.1 Final Test Metrics

In [ ]:
y_pred_final = final_pipeline.predict(X_test)

final_scores = {
    'R2': r2_score(y_test, y_pred_final),
    'MAE': mean_absolute_error(y_test, y_pred_final),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_final))
}

print(pd.DataFrame([final_scores]).to_string(index=False))

### 9.2 Metrics Explained

- **R² Score:** Model kitna variation explain kar raha hai. `0.932` ka matlab model ne mental health score ke pattern ka **93.2%** explain kiya.
- **MAE (Mean Absolute Error):** Average prediction error, original score points mein. `0.258` ka matlab prediction average **0.26 score points** off hai.
- **RMSE (Root Mean Squared Error):** MAE jaisa, lekin bade errors ko zyada penalty deta hai. Agar RMSE MAE se noticeably bada ho to model kuch predictions mein bade mistakes kar raha hai.

### 9.3 Cross-Validation of Final Model

Test set ek hi split hai. Final model ka CV score zyada honest estimate deta hai ke model different splits par kaisa perform karta hai.

In [ ]:
final_cv = cross_validate(
    final_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=[
        'r2',
        'neg_mean_absolute_error',
        'neg_root_mean_squared_error'
    ]
)

print(f"CV R2   : {final_cv['test_r2'].mean():.4f}")
print(f"CV MAE  : {-final_cv['test_neg_mean_absolute_error'].mean():.4f}")
print(f"CV RMSE : {-final_cv['test_neg_root_mean_squared_error'].mean():.4f}")

### 9.4 Actual vs Predicted

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=y_pred_final, alpha=0.5)

line_min = min(y_test.min(), y_pred_final.min())
line_max = max(y_test.max(), y_pred_final.max())
plt.plot([line_min, line_max], [line_min, line_max], 'r--')

plt.title(f'Actual vs Predicted — {final_model_name}')
plt.xlabel('Actual Mental Health Score')
plt.ylabel('Predicted Mental Health Score')
plt.show()

## 10. Save Final Pipeline as Compressed `.pkl.gz`

Ab final pipeline ko `pickle` se save kar rahe hain, lekin `gzip` compression ke saath. Isse model bilkul same rehta hai (lossless) aur file size **~46 MB se ~9.4 MB** ho jati hai. FastAPI mein `gzip.open()` + `pickle.load()` se load kar sakte hain.

> `pickle` Python-specific hai. Agar model sirf Python/FastAPI mein use hoga to yeh perfect hai. Cross-language deployment ke liye ONNX better hota hai.

In [ ]:
with gzip.open('mental_health_pipeline.pkl.gz', 'wb', compresslevel=6) as f:
    pickle.dump(final_pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved final pipeline ({final_model_name}) as mental_health_pipeline.pkl.gz")

## 11. Workflow Summary

| Step | What Happened |
|---|---|
| Duplicate removal | 2 exact duplicates removed |
| Negative fix | Negative Physical_Activity_Hours → 0 |
| Train/Test split | 3998 train / 1000 test |
| Numeric branch | Median impute → StandardScaler |
| Ordinal branch | Most-frequent impute → OrdinalEncoder with order |
| Nominal branch | Most-frequent impute → OneHotEncoder with rare-category grouping |
| Model pipelines | Linear Regression + Random Forest compared |
| Cross-validation | 5-fold CV for reliable comparison |
| Hyperparameter tuning | RandomizedSearchCV on Random Forest |
| Model evaluation | R2, MAE, RMSE on untouched test set |
| Deployment | Final pipeline saved as compressed `.pkl.gz` |

**Leakage avoided:** Test data ko kabhi bhi imputer, encoder, scaler, model fit, ya hyperparameter tuning ke liye use nahi kiya gaya.

## Next Step

- Feature importance analysis
- FastAPI deployment using `mental_health_pipeline.pkl.gz`
- Optional: SHAP-based model explainability